<a href="https://colab.research.google.com/github/pichu2707/corr-causal-enae/blob/main/Post_ENAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Ejemplos básicos de Covarianza
Códigos de ejemplo de covarianzas para ver los distintos resulatado que podemos obtener para revisar donde está estas relaciones.

$$\operatorname{cov}(x,y)=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{n}$$



In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf

In [ ]:
x = [2, 4, 6, 8, 10]
y = [65, 70, 75, 80, 85]

In [ ]:
 # Calculamos la media de cada uno
 x_mean = np.mean(x)
 y_mean = np.mean(y)

In [ ]:
#calculamos la covarianza entre ellos

covarianza_positiva = np.sum((x - x_mean) * (y - y_mean)) / (len(x) - 1)


In [ ]:
print(f"El valor de la covarianza es de: {covarianza_positiva}")

El valor de la covarianza es de: 25.0


In [ ]:
x1 = [1, 2, 3, 4, 5]
y1 = [10, 9, 8, 7, 6]

In [ ]:
x1_mean = np.mean(x1)
y1_mean = np.mean(y1)

In [ ]:
covarianza_negativa = np.sum((x1 - x1_mean) * (y1 - y1_mean)) / (len(x1) - 1)

In [ ]:
print(f"El valor de la covarianza es: {covarianza_negativa}")

El valor de la covarianza es: -2.5


## Ejemplo básico de correlación de Pearson
En este código vamos a mostrar un ejemplo de la correlación de Pearson

$$\rho x,y=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{Var(X)Var(Y)}}$$


In [ ]:
# Calculando la correlación de la matriz
matriz_correlacion = np.corrcoef(x, y)

In [ ]:
corr_pearson = matriz_correlacion[0, 1]

In [ ]:
print(f"La correlación entre X e Y con pearson es: {corr_pearson}")

La correlación entre X e Y con pearson es: 1.0


In [ ]:
x2 = np.array([2, 4, 6, 8, 10])
y2 = np.array([65, 70, 75, 80, 85])
coef_pearson, valor_p = pearsonr(x2, y2)

In [ ]:
print(f"El coeficiente de correlación de Pearson es: {coef_pearson}")
print(f"El valor p es: {valor_p}")

El coeficiente de correlación de Pearson es: 1.0
El valor p es: 0.0


## Ejemplo de correlación de Spearman
Código de ejemplo de correlación de Spearman

$$\rho=1-\frac{6\sum{d_i^2}}{n(n^2-1)}$$

In [ ]:
# Datos que usamos de relación monónotan no lineal (y=x^2)
x_spearman = np.array([2, 4, 6, 8, 10])
y_spearman = np.array([1, 4, 9, 16, 25])

In [ ]:
#Calculamos la correalación de Spearman y el valor p asociado
coef_spearman, valor_p = spearmanr(x_spearman, y_spearman)

In [ ]:
print(f"El coeficiente de correlación de Spearman es: {coef_spearman}")
print(f"El valor p es: {valor_p}")

El coeficiente de correlación de Spearman es: 0.9999999999999999
El valor p es: 1.4042654220543672e-24


## Ejemplo de inferencia Causal con DiD

In [2]:
#Fimjamos la semilla para repoducibilidad
np.random.seed(42)

#Número total de unidades, podría ser cientes o regociones
n_unidades=200

In [3]:
# Asignaos aleatoriamente la mitad de las unidades al grupo de tratamiento 1 y la otra mitad al grupo de control 0
grupo = np.concatenate([np.zeros(n_unidades//2), np.ones(n_unidades//2)])

In [8]:
# Creamos una base de datos en formato panel (cada unidad se observa en dos periodos: pre y post tratamiento)
datos = []
for i in range(n_unidades):
  for tiempo in [0, 1]: # 0: pre tratamiento, 1 post tratamiento
    datos.append({
        "unidad": i,
        'grupo': grupo[i],
        'tiempo': tiempo
    })

df = pd.DataFrame(datos)
print(df.head(10))

   unidad  grupo  tiempo
0       0    0.0       0
1       0    0.0       1
2       1    0.0       0
3       1    0.0       1
4       2    0.0       0
5       2    0.0       1
6       3    0.0       0
7       3    0.0       1
8       4    0.0       0
9       4    0.0       1


Generamos la variable de reultado (outcome)
Se asume que:
* Un valor base de 10
* Efecto del tiempo: en el periodo post se suma 2 unidades a todos
* Efecto del tratamiento: solo se aplica en el grupo tratado en el periodo post y su efecto es de 3 unidades
* Se añade ruido aleatoriamente


In [10]:
df['ruido'] = np.random.normal(0, 1, len(df))
df['resultado']= (
    10+
    2*df['tiempo']+ #Efecto de pasar de pre a post en general
    0*df['grupo']+ # Efecto base del grupo (se hace 0 para este ejemplo)
    3*df['grupo']*df['tiempo']+ # Efecto extra solo en el grupo de tratamiento en el periodo post
    df['ruido']
)

print(50*"=")
print("Datos de ejemplo:")
print(50*"=")

print(df.head(10))

Datos de ejemplo:
   unidad  grupo  tiempo     ruido  resultado
0       0    0.0       0 -1.594428   8.405572
1       0    0.0       1 -0.599375  11.400625
2       1    0.0       0  0.005244  10.005244
3       1    0.0       1  0.046981  12.046981
4       2    0.0       0 -0.450065   9.549935
5       2    0.0       1  0.622850  12.622850
6       3    0.0       0 -1.067620   8.932380
7       3    0.0       1 -0.142379  11.857621
8       4    0.0       0  0.120296  10.120296
9       4    0.0       1  0.514439  12.514439


Realizamos el análisis de diferencias en diferencias.
El modelo estima la variable 'resultado' en función de:
* 'grupo': Diferencia base entre grupos.
* 'tiempo': Diferencia general entre periodos (pre vs. post).
* 'grupo:tiempo': La interacción, que captura el efecto causal del tratamiento.

In [11]:
modelo = smf.ols("resultado ~ grupo + tiempo + grupo:tiempo", data=df).fit()

print("\nResumen del modelo de diferencias en diferencias:")
print(modelo.summary())


Resumen del modelo de diferencias en diferencias:
                            OLS Regression Results                            
Dep. Variable:              resultado   R-squared:                       0.815
Model:                            OLS   Adj. R-squared:                  0.814
Method:                 Least Squares   F-statistic:                     583.3
Date:                Mon, 14 Apr 2025   Prob (F-statistic):          6.88e-145
Time:                        21:09:44   Log-Likelihood:                -568.43
No. Observations:                 400   AIC:                             1145.
Df Residuals:                     396   BIC:                             1161.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------